# 강의 04 · 실습 2 — 체크포인트와 사람 개입 · (5) 고난도 II — 반려와 다시 쓰기

## 1. 문제상황

- 여행사 고객센터 담당자는 초안을 검토할 때 승인만 하는 것이 아니라 반려도 합니다.
- 지금의 처리 흐름에서는 담당자가 초안을 직접 고쳐 써야 발송되는데, 담당자는 「수수료 금액을 명시할 것」처럼 고칠 방향만 말하고 다시 쓰는 일은 프로그램이 하기를 바랍니다.
- 반려된 초안이 다시 담당자에게 오는 횟수도 기록해야 합니다. 같은 초안이 계속 반려되면 사람이 직접 써야 하기 때문입니다.
- 같은 고객의 앞선 대화와 체크포인트는 그대로 유지되어야 합니다.

## 2. 문제와 목표

- **문제**: 담당자가 초안을 승인하거나 직접 고쳐 쓰는 두 가지 답만 할 수 있어, 「이 방향으로 다시 써라」는 반려를 처리할 자리가 없습니다.
- **목표**
  - 초안을 만든 뒤 발송 앞에서 멈춰 담당자의 답을 받습니다.
    - 담당자의 답 둘: 승인이면 문자열 「승인」, 반려이면 사유를 담은 딕셔너리
  - 담당자가 승인하면 발송하고, 반려 사유를 주면 그 사유를 반영해 초안을 다시 쓴 뒤 다시 담당자에게 묻는 처리 흐름을 만듭니다.
  - 다시 쓴 횟수를 상태에 기록하고, 메일마다 새 입력을 넣을 때 반려 사유는 빈 문자열, 다시 쓴 횟수는 0으로 초기화합니다.
  - 같은 `thread_id` 아래 대화 기록과 체크포인트, 오래된 대화의 요약은 그대로 유지합니다.
    - 사람이 정한 값: 원문으로 남기는 최근 메시지 수 4, 요약 지시문 「다음 여행사 고객센터 대화를 고객이 문의한 내용과 상담원이 안내한 처리 중심으로 두 문장 이내 한국어로 요약한다. 새 정보를 지어내지 않는다.」, `thread_id` `customer-9350`
    - 메일 두 통과 첫 메일의 반려 사유는 「6. 코드 — 스텝바이스텝」 단계 0에 주어져 있습니다.
- **목표 달성 여부의 판정 기준**: 같은 `thread_id`로 메일 두 통을 넣고, 첫 통은 담당자가 한 번 반려한 뒤 승인하고 두 번째 통은 바로 승인했을 때,
  - 첫 통에서 그래프가 발송 앞에서 두 번 멈추고, 담당자에게 간 내용에는 초안과 다시 쓴 횟수가 들어 있고, 두 번째 초안이 반려 사유를 반영하며, 다시 쓴 횟수가 1로 기록되고, 발송은 승인 뒤 한 번만 일어나며,
  - 두 번째 통은 한 번 멈추고 다시 쓴 횟수가 0인 것을 실행 결과에서 확인합니다.
    - 출력 줄에는 「[멈춘 지점]」「[첫 번째 초안]」「[담당자에게 간 내용]」「[재개 후]」「[반려 뒤 멈춘 지점]」「[발송]」「[대화 기록]」 표지를 붙이고, 멈춘 지점은 `next = (노드 이름,)` 형태로 출력합니다.
  - 담당자의 방침은 대본으로 미리 넣습니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex02_s5_diagram.svg)

## 4. 단계별 요구사항

(「3. 워크플로우 다이어그램」과 「2. 문제와 목표」의 목표를 보고 요구사항을 번호 목록으로 직접 씁니다. 상태의 키, 노드마다 읽는 키와 쓰는 키, 엣지의 종류, 실행 순서를 적습니다.)

## 5. 코드 골격

이 실습의 코드 골격을 직접 세웁니다. 단계 · 하는 일 · 사용하는 코드 · 대응하는 요구사항 네 칸 표로 적습니다.

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 모델을 준비합니다. 체크포인트를 저장할 파일 위치도 여기서 정합니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- 체크포인트 파일은 실행할 때마다 새 임시 폴더에 만듭니다. 지난 실행의 저장 기록이 이번 실행에 섞이지 않게 하기 위해서입니다.
- 대화 기록 출력은 `show_messages(msgs)`로 합니다.

In [ ]:
import sqlite3
import tempfile
from pathlib import Path
import os

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import AIMessage, HumanMessage, RemoveMessage, SystemMessage
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import Command, interrupt

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")

DB_PATH = Path(tempfile.mkdtemp(prefix="lec04_ex02_")) / "checkpoint.db"   # 체크포인트를 저장할 파일


def show_messages(msgs: list) -> None:
    """대화 기록을 메시지 종류와 앞부분 글자만 한 줄씩 출력한다."""
    for m in msgs:
        print(f"      [{type(m).__name__}] {str(m.content)[:90]}")


print("모델 준비를 마쳤습니다. 체크포인트 파일:", DB_PATH.name)


# 주어진 자료
EMAILS = [
    "출발일을 이틀 뒤로 바꾸고 싶습니다. 비용이 얼마나 드는지 알려 주세요.",
    "알려 주신 대로 진행해 주세요. 변경 뒤 새 항공권은 어떻게 받나요?",
]
NEW_INPUT = {"feedback": "", "retries": 0, "sent": False}   # 메일마다 초기화하는 키
REJECT_REASON = "변경 수수료 금액(3만 원)을 반드시 명시할 것."   # 1번 메일 첫 초안에 담당자가 주는 반려 사유 (대본)


In [ ]:
# 여기에 코드를 작성합니다. 단계마다 셀을 나누어 작성합니다.


## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 1번 메일에서 `next = ('human_review',)`가 두 번 출력됩니다. 첫 번째 멈춤 뒤 반려하면 `[발송]` 없이 다시 멈추고, 두 번째 초안이 반려 사유(수수료 금액 명시)를 반영합니다.
2. 1번 메일의 두 번째 멈춤에서 담당자에게 간 내용의 다시 쓴 횟수가 1이고, 승인 뒤 `[발송]`이 한 번만 출력됩니다.
3. 2번 메일은 한 번 멈추고 바로 승인되어 다시 쓴 횟수가 0이며, `[대화 기록]`에 1번 메일과 그 답장이 남아 있습니다.
